In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except:
    pass

In [ ]:
import os

drive_folder_name = "ProductVoiceAnalytics"
os.chdir(f'/content/drive/MyDrive/Projects/GitHub/{drive_folder_name}')

!pwd

# Notebook 04 — Model Comparison (KEY)


Compare TF-IDF + LR (baseline) vs DistilBERT (fine-tuned) on the held-out test set.

In [ ]:
# import os
# if os.path.basename(os.getcwd()) == 'notebooks':
#     os.chdir('..')

In [ ]:
import time
import numpy as np
import pandas as pd
import joblib
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.pipeline import Pipeline

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, Dataset

from src.config import (
    PROCESSED_SAMPLE_PATH,
    DISTILBERT_MAX_LEN,
    DISTILBERT_BATCH_SIZE
)

## 1. Load Test Data

In [ ]:
df = pd.read_csv(PROCESSED_SAMPLE_PATH)
df['clean_text'] = df['clean_text'].fillna('').astype(str)
# drop any rows where clean_text is empty — can happen if reviewText was malformed
df = df[df['clean_text'].str.strip().astype(bool)].reset_index(drop=True)


_, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

X_test = test_df['clean_text']
y_test = test_df['label']

print(f'Test set: {len(test_df)} rows')
print(test_df['label'].value_counts())

## 2. Evaluate TF-IDF + Logistic Regression

In [ ]:
# tfidf_pipeline = joblib.load('models/tfidf_pipeline.pkl')
vectorizer = joblib.load('models/tfidf_vectorizer.pkl')
lr_model   = joblib.load('models/lr_model.pkl')

tfidf_pipeline = Pipeline([('tfidf', vectorizer), ('lr', lr_model)])

start = time.time()
tfidf_preds = tfidf_pipeline.predict(X_test)
tfidf_elapsed = time.time() - start

tfidf_speed = (tfidf_elapsed / len(X_test)) * 1000

tfidf_acc = accuracy_score(y_test, tfidf_preds)
tfidf_f1  = f1_score(y_test, tfidf_preds, average='macro')

print(classification_report(y_test, tfidf_preds, target_names=['negative', 'neutral', 'positive']))
print(f'Speed: {tfidf_speed:.3f}s per 1000 reviews')

## 3. Evaluate DistilBERT

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device('mps')
    elif torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')

device = get_device()
print(f'Using device: {device}')

In [ ]:
print(f'Using device: {device}')

tokenizer = DistilBertTokenizerFast.from_pretrained('models/distilbert/')
model     = DistilBertForSequenceClassification.from_pretrained('models/distilbert/')
model     = model.to(device)
model.eval()

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_len,
            return_tensors='pt'
        )

    def __len__(self):
        return self.encodings['input_ids'].shape[0]

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        return item

In [ ]:
test_dataset = ReviewDataset(X_test.tolist(), tokenizer, DISTILBERT_MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=DISTILBERT_BATCH_SIZE)

bert_preds = []

start = time.time()

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs        = model(input_ids=input_ids, attention_mask=attention_mask)
        predictions    = torch.argmax(outputs.logits, dim=1)
        bert_preds.extend(predictions.cpu().numpy())

bert_elapsed = time.time() - start
bert_speed   = (bert_elapsed / len(X_test)) * 1000

bert_acc = accuracy_score(y_test, bert_preds)
bert_f1  = f1_score(y_test, bert_preds, average='macro')

print(classification_report(y_test, bert_preds, target_names=['negative', 'neutral', 'positive']))
print(f'Speed: {bert_speed:.3f}s per 1000 reviews')

## 4. Comparison Table

In [ ]:
results = pd.DataFrame({
    'Model':        ['TF-IDF + LR', 'DistilBERT'],
    'Accuracy':     [tfidf_acc, bert_acc],
    'Macro-F1':     [tfidf_f1, bert_f1],
    'Speed (s/1k)': [tfidf_speed, bert_speed],
})

results = results.round(4)

print(results.to_string(index=False))

## 5. Save Best Model Reference

In [ ]:
# best_idx        = results['Macro-F1'].idxmax()
# best_model_name = 'distilbert' if best_idx == 1 else 'tfidf_lr'

# os.makedirs('models', exist_ok=True)
# ref_path = 'models/best_model_ref.txt'

# with open(ref_path, 'w') as f:
#     f.write(best_model_name)

# print(f'Best model: {best_model_name} — reference saved to {ref_path}')